# Benchmark Comparatif : Compression PCA appliquée aux 3 méthodes ANN

**Projet** : Comparative ANN Methods — Compression par Réduction de Dimension

**Objectif** : Évaluer l'impact d'une **compression par PCA** (128D → 32D) sur les performances des trois méthodes de recherche approximative de voisins les plus proches :
- **HNSW** (Hierarchical Navigable Small World)
- **LSH** (Locality Sensitive Hashing)
- **IVF-PQ** (Inverted File + Product Quantization)

**Dataset** : SIFT1M (1 million de vecteurs, dimension = 128)

---

## Hypothèses testées

| Méthode | Effet attendu de la PCA | Justification |
|---|---|---|
| **HNSW** | Recall légèrement en baisse | HNSW gère bien la haute dim, la PCA retire un peu d'info |
| **LSH** | Recall fortement en hausse 📈 | LSH souffre du "curse of dimensionality", la PCA le sauve |
| **IVF-PQ** | Recall stable ou légère baisse | IVF-PQ a déjà sa propre compression |

Dans tous les cas, on attend :
- **Mémoire divisée par ~4** (128D × 4 o → 32D × 4 o)
- **Latence réduite** (calcul de distance 4× plus rapide)
- **Build time réduit** (moins de données à traiter)

---

## Plan du notebook

1. Setup & chargement SIFT1M
2. Fondements de la PCA : entraînement et analyse
3. Benchmark HNSW (avec vs sans PCA)
4. Benchmark LSH (avec vs sans PCA)
5. Benchmark IVF-PQ (avec vs sans PCA)
6. Synthèse comparative et visualisations
7. Conclusions


## 1. Setup & Configuration

### 1.1 Imports et paramètres globaux


In [ ]:
# Installation (décommenter si nécessaire)
# !pip install faiss-cpu scikit-learn numpy matplotlib seaborn psutil pandas -q


In [ ]:
import numpy as np
import faiss
import time
import psutil
import gc
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from sklearn.decomposition import PCA

# Style des graphiques
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 7)

# Reproductibilité
np.random.seed(42)

print(f"✓ FAISS version : {faiss.__version__}")
print(f"✓ NumPy version : {np.__version__}")


In [ ]:
# ============================================
# CONSTANTES GLOBALES
# ============================================

# Paramètres dataset
DIM_ORIGINAL = 128           # Dimension SIFT1M
DIM_PCA = 32                 # Dimension cible après PCA (÷4)
K = 10                       # Nb voisins à chercher
NUM_QUERIES = 100            # Nb de requêtes de test

# Chemins
DATA_PATH = "./data/"        # ⚠️ Ajuste selon ton arborescence
RESULTS_PATH = "./results/"

# Hyperparamètres "sweet spot" pour chaque méthode
# (sélectionnés sur la base des benchmarks individuels)
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 40
HNSW_EF_SEARCH = 100

LSH_NBITS = 1024

IVFPQ_NLIST = 1024
IVFPQ_M_SUBQ = 8             # Doit diviser la dimension (8 divise 128 ET 32 ✓)
IVFPQ_NBITS = 8
IVFPQ_NPROBE = 10
IVFPQ_TRAIN_SIZE = 100_000

Path(RESULTS_PATH).mkdir(parents=True, exist_ok=True)

print("✓ Configuration :")
print(f"  - Dimension originale     : {DIM_ORIGINAL}")
print(f"  - Dimension après PCA     : {DIM_PCA}  (ratio de compression = {DIM_ORIGINAL/DIM_PCA:.1f}×)")
print(f"  - K (voisins)             : {K}")
print(f"  - Nb requêtes             : {NUM_QUERIES}")


### 1.2 Chargement du dataset SIFT1M


In [ ]:
def load_fvecs(filename):
    """Charge un fichier .fvecs (vecteurs float32)."""
    with open(filename, 'rb') as f:
        dim = np.fromfile(f, dtype=np.int32, count=1)[0]
        f.seek(0)
        data = np.fromfile(f, dtype=np.int32)
        vec_size = 1 + dim
        n_vectors = len(data) // vec_size
        data = data.reshape(n_vectors, vec_size)
        return data[:, 1:].view(np.float32)

def load_ivecs(filename):
    """Charge un fichier .ivecs (vecteurs int32)."""
    with open(filename, 'rb') as f:
        dim = np.fromfile(f, dtype=np.int32, count=1)[0]
        f.seek(0)
        data = np.fromfile(f, dtype=np.int32)
        vec_size = 1 + dim
        n_vectors = len(data) // vec_size
        data = data.reshape(n_vectors, vec_size)
        return data[:, 1:]


In [ ]:
# Vérification préalable
required = ['sift_base.fvecs', 'sift_query.fvecs', 'sift_groundtruth.ivecs']
missing = [f for f in required if not os.path.exists(os.path.join(DATA_PATH, f))]

if missing:
    raise FileNotFoundError(
        f"Fichiers manquants dans '{DATA_PATH}': {missing}\n"
        f"Répertoire courant : {os.getcwd()}\n"
        f"Télécharge SIFT1M : ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz"
    )

print("📂 Chargement SIFT1M...")

base_vectors = load_fvecs(f"{DATA_PATH}sift_base.fvecs")
print(f"   ✓ base_vectors  : {base_vectors.shape}  dtype={base_vectors.dtype}")

query_vectors = load_fvecs(f"{DATA_PATH}sift_query.fvecs")[:NUM_QUERIES]
print(f"   ✓ query_vectors : {query_vectors.shape}")

ground_truth = load_ivecs(f"{DATA_PATH}sift_groundtruth.ivecs")[:NUM_QUERIES, :K]
print(f"   ✓ ground_truth  : {ground_truth.shape}")

# ⚠️ Important : on garde le ground truth ORIGINAL (128D)
# Toutes les méthodes, même avec PCA, seront évaluées contre ce ground truth.
# Cela mesure la perte totale (PCA + ANN) qui est ce qui compte pour l'utilisateur.

print("\n✅ Dataset chargé avec succès !")


## 2. Entraînement et analyse de la PCA

### 2.1 Principe

La PCA cherche les **axes de variance maximale** dans les données. On projette les vecteurs 128D sur un sous-espace 32D formé par les 32 composantes principales.

**Étapes :**
1. Centrer les données (soustraire la moyenne)
2. Calculer la matrice de covariance
3. Décomposition en valeurs propres → axes principaux classés par variance
4. Projection sur les `d' = 32` premiers axes

### 2.2 Point méthodologique crucial

⚠️ **La PCA s'entraîne UNIQUEMENT sur `base_vectors`.**

Les vecteurs de requête doivent ensuite être projetés avec les **mêmes axes**, sinon on introduit un biais (data leakage). C'est la règle classique `fit` / `transform` du machine learning.


In [ ]:
# ============================================
# ENTRAÎNEMENT DE LA PCA
# ============================================

print(f"🎓 Entraînement PCA : {DIM_ORIGINAL}D → {DIM_PCA}D")
print(f"   Base d'entraînement : {base_vectors.shape[0]:,} vecteurs")

start = time.time()
pca = PCA(n_components=DIM_PCA, random_state=42)
pca.fit(base_vectors)
pca_fit_time = time.time() - start

print(f"\n✓ PCA entraînée en {pca_fit_time:.2f}s")


In [ ]:
# ============================================
# PROJECTION DES VECTEURS
# ============================================

print("📉 Projection des vecteurs dans l'espace réduit...")

start = time.time()
base_pca = pca.transform(base_vectors).astype('float32')
base_proj_time = time.time() - start
print(f"   ✓ base_pca  : {base_pca.shape}  (projection en {base_proj_time:.2f}s)")

start = time.time()
query_pca = pca.transform(query_vectors).astype('float32')
query_proj_time = time.time() - start
print(f"   ✓ query_pca : {query_pca.shape}  (projection en {query_proj_time:.3f}s)")

pca_total_offline_time = pca_fit_time + base_proj_time + query_proj_time
print(f"\n⏱️  Temps total PCA (offline) : {pca_total_offline_time:.2f}s")


### 2.3 Analyse de la variance expliquée

**Indicateur clé** : combien d'information est-ce qu'on conserve avec 32 composantes sur 128 ?


In [ ]:
# ============================================
# VARIANCE EXPLIQUÉE
# ============================================

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print(f"📊 Variance expliquée :")
print(f"   Par la 1ère composante  : {explained_var[0]*100:.2f}%")
print(f"   Par les 10 premières    : {cumulative_var[9]*100:.2f}%")
print(f"   Par les 20 premières    : {cumulative_var[19]*100:.2f}%")
print(f"   Par les {DIM_PCA} premières (total) : {cumulative_var[-1]*100:.2f}%")

print(f"\n💡 Interprétation : en passant de {DIM_ORIGINAL}D à {DIM_PCA}D, on conserve "
      f"{cumulative_var[-1]*100:.1f}% de l'information structurelle.")


In [ ]:
# ============================================
# VISUALISATION : Variance expliquée
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# (a) Variance par composante (bars)
axes[0].bar(range(1, DIM_PCA+1), explained_var * 100, color='#2E86AB', alpha=0.8)
axes[0].set_xlabel('N° de composante principale', fontsize=12)
axes[0].set_ylabel('Variance expliquée (%)', fontsize=12)
axes[0].set_title(f'Variance par composante (top {DIM_PCA})', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# (b) Variance cumulée (ligne)
axes[1].plot(range(1, DIM_PCA+1), cumulative_var * 100,
             marker='o', linewidth=2.5, markersize=6, color='#A23B72')
axes[1].axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='Seuil 90%')
axes[1].axhline(y=95, color='red', linestyle='--', alpha=0.7, label='Seuil 95%')
axes[1].set_xlabel('Nombre de composantes retenues', fontsize=12)
axes[1].set_ylabel('Variance cumulée (%)', fontsize=12)
axes[1].set_title('Variance cumulée expliquée', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}pca_variance_explained.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Fonctions utilitaires de benchmark


In [ ]:
def calculate_recall(I_pred, I_gt, k=K):
    """Recall@K moyen."""
    recalls = []
    for i in range(len(I_pred)):
        inter = np.intersect1d(I_pred[i][:k], I_gt[i][:k])
        recalls.append(len(inter) / k)
    return float(np.mean(recalls))

def get_memory_mb():
    """Mémoire utilisée par le processus en Mo."""
    return psutil.Process().memory_info().rss / (1024 * 1024)

def measure_search(index, queries, k=K, n_runs=3):
    """Mesure latence moyenne (ms) + QPS + renvoie les résultats."""
    # Warmup
    index.search(queries[:min(10, len(queries))], k)
    times = []
    for _ in range(n_runs):
        t0 = time.time()
        D, I = index.search(queries, k)
        times.append(time.time() - t0)
    avg_time = np.mean(times)
    latency_ms = (avg_time / len(queries)) * 1000
    qps = len(queries) / avg_time
    return latency_ms, qps, I

print("✓ Fonctions utilitaires définies")


## 4. Benchmark HNSW : avec vs sans PCA

Hyperparamètres fixés : M=32, efConstruction=40, efSearch=100


In [ ]:
# ============================================
# HNSW SANS PCA (baseline 128D)
# ============================================

print("="*60)
print(f"HNSW — SANS PCA (dim = {DIM_ORIGINAL})")
print("="*60)

mem_before = get_memory_mb()

# Construction
t0 = time.time()
index_hnsw = faiss.IndexHNSWFlat(DIM_ORIGINAL, HNSW_M)
index_hnsw.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
index_hnsw.add(base_vectors)
build_time_hnsw = time.time() - t0

mem_index_hnsw = get_memory_mb() - mem_before

# Recherche
index_hnsw.hnsw.efSearch = HNSW_EF_SEARCH
latency_hnsw, qps_hnsw, I_hnsw = measure_search(index_hnsw, query_vectors)
recall_hnsw = calculate_recall(I_hnsw, ground_truth)

print(f"  Build time : {build_time_hnsw:.2f}s")
print(f"  Mémoire    : {mem_index_hnsw:.1f} MB")
print(f"  Recall@10  : {recall_hnsw:.4f}")
print(f"  Latence    : {latency_hnsw:.4f} ms/query")
print(f"  QPS        : {qps_hnsw:.1f}")

del index_hnsw
gc.collect()


In [ ]:
# ============================================
# HNSW AVEC PCA (32D)
# ============================================

print("="*60)
print(f"HNSW — AVEC PCA (dim = {DIM_PCA})")
print("="*60)

mem_before = get_memory_mb()

# Construction
t0 = time.time()
index_hnsw_pca = faiss.IndexHNSWFlat(DIM_PCA, HNSW_M)
index_hnsw_pca.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
index_hnsw_pca.add(base_pca)
build_time_hnsw_pca = time.time() - t0

mem_index_hnsw_pca = get_memory_mb() - mem_before

# Recherche
index_hnsw_pca.hnsw.efSearch = HNSW_EF_SEARCH
latency_hnsw_pca, qps_hnsw_pca, I_hnsw_pca = measure_search(index_hnsw_pca, query_pca)

# ⚠️ Recall vs ground truth ORIGINAL (128D)
recall_hnsw_pca = calculate_recall(I_hnsw_pca, ground_truth)

print(f"  Build time : {build_time_hnsw_pca:.2f}s")
print(f"  Mémoire    : {mem_index_hnsw_pca:.1f} MB")
print(f"  Recall@10  : {recall_hnsw_pca:.4f}")
print(f"  Latence    : {latency_hnsw_pca:.4f} ms/query")
print(f"  QPS        : {qps_hnsw_pca:.1f}")

del index_hnsw_pca
gc.collect()

# Delta
print(f"\n📊 Delta (PCA vs original) :")
print(f"   Recall    : {recall_hnsw_pca - recall_hnsw:+.4f}  ({(recall_hnsw_pca/recall_hnsw - 1)*100:+.1f}%)")
print(f"   Latence   : {(latency_hnsw_pca/latency_hnsw - 1)*100:+.1f}%")
print(f"   Mémoire   : {(mem_index_hnsw_pca/mem_index_hnsw - 1)*100:+.1f}%")
print(f"   Build     : {(build_time_hnsw_pca/build_time_hnsw - 1)*100:+.1f}%")


## 5. Benchmark LSH : avec vs sans PCA

Hyperparamètre fixé : nbits = 1024

💡 **C'est sur cette méthode qu'on s'attend au plus gros gain** : LSH souffre de la haute dimension, la PCA devrait améliorer significativement son recall.


In [ ]:
# ============================================
# LSH SANS PCA
# ============================================

print("="*60)
print(f"LSH — SANS PCA (dim = {DIM_ORIGINAL}, nbits = {LSH_NBITS})")
print("="*60)

mem_before = get_memory_mb()

t0 = time.time()
index_lsh = faiss.IndexLSH(DIM_ORIGINAL, LSH_NBITS)
if not index_lsh.is_trained:
    index_lsh.train(base_vectors)
index_lsh.add(base_vectors)
build_time_lsh = time.time() - t0

mem_index_lsh = get_memory_mb() - mem_before

latency_lsh, qps_lsh, I_lsh = measure_search(index_lsh, query_vectors)
recall_lsh = calculate_recall(I_lsh, ground_truth)

print(f"  Build time : {build_time_lsh:.2f}s")
print(f"  Mémoire    : {mem_index_lsh:.1f} MB")
print(f"  Recall@10  : {recall_lsh:.4f}")
print(f"  Latence    : {latency_lsh:.4f} ms/query")
print(f"  QPS        : {qps_lsh:.1f}")

del index_lsh
gc.collect()


In [ ]:
# ============================================
# LSH AVEC PCA
# ============================================

print("="*60)
print(f"LSH — AVEC PCA (dim = {DIM_PCA}, nbits = {LSH_NBITS})")
print("="*60)

mem_before = get_memory_mb()

t0 = time.time()
index_lsh_pca = faiss.IndexLSH(DIM_PCA, LSH_NBITS)
if not index_lsh_pca.is_trained:
    index_lsh_pca.train(base_pca)
index_lsh_pca.add(base_pca)
build_time_lsh_pca = time.time() - t0

mem_index_lsh_pca = get_memory_mb() - mem_before

latency_lsh_pca, qps_lsh_pca, I_lsh_pca = measure_search(index_lsh_pca, query_pca)
recall_lsh_pca = calculate_recall(I_lsh_pca, ground_truth)

print(f"  Build time : {build_time_lsh_pca:.2f}s")
print(f"  Mémoire    : {mem_index_lsh_pca:.1f} MB")
print(f"  Recall@10  : {recall_lsh_pca:.4f}")
print(f"  Latence    : {latency_lsh_pca:.4f} ms/query")
print(f"  QPS        : {qps_lsh_pca:.1f}")

del index_lsh_pca
gc.collect()

print(f"\n📊 Delta (PCA vs original) :")
print(f"   Recall    : {recall_lsh_pca - recall_lsh:+.4f}  ({(recall_lsh_pca/recall_lsh - 1)*100:+.1f}%)")
print(f"   Latence   : {(latency_lsh_pca/latency_lsh - 1)*100:+.1f}%")
print(f"   Mémoire   : {(mem_index_lsh_pca/mem_index_lsh - 1)*100:+.1f}%")


## 6. Benchmark IVF-PQ : avec vs sans PCA

Hyperparamètres fixés : nlist = 1024, m = 8, nbits = 8, nprobe = 10

⚠️ **Note sur `m`** : le paramètre `m` de Product Quantization doit diviser la dimension des vecteurs. 8 divise à la fois 128 et 32 → on utilise la même valeur dans les deux cas.


In [ ]:
# ============================================
# IVF-PQ SANS PCA
# ============================================

print("="*60)
print(f"IVF-PQ — SANS PCA (dim = {DIM_ORIGINAL})")
print("="*60)

mem_before = get_memory_mb()

train_data = base_vectors[:IVFPQ_TRAIN_SIZE].copy()

t0 = time.time()
quantizer = faiss.IndexFlatL2(DIM_ORIGINAL)
index_ivfpq = faiss.IndexIVFPQ(quantizer, DIM_ORIGINAL, IVFPQ_NLIST, IVFPQ_M_SUBQ, IVFPQ_NBITS)
index_ivfpq.train(train_data)
index_ivfpq.add(base_vectors)
build_time_ivfpq = time.time() - t0

mem_index_ivfpq = get_memory_mb() - mem_before

index_ivfpq.nprobe = IVFPQ_NPROBE
latency_ivfpq, qps_ivfpq, I_ivfpq = measure_search(index_ivfpq, query_vectors)
recall_ivfpq = calculate_recall(I_ivfpq, ground_truth)

print(f"  Build time : {build_time_ivfpq:.2f}s")
print(f"  Mémoire    : {mem_index_ivfpq:.1f} MB")
print(f"  Recall@10  : {recall_ivfpq:.4f}")
print(f"  Latence    : {latency_ivfpq:.4f} ms/query")
print(f"  QPS        : {qps_ivfpq:.1f}")

del index_ivfpq, quantizer, train_data
gc.collect()


In [ ]:
# ============================================
# IVF-PQ AVEC PCA
# ============================================

print("="*60)
print(f"IVF-PQ — AVEC PCA (dim = {DIM_PCA})")
print("="*60)

mem_before = get_memory_mb()

train_data_pca = base_pca[:IVFPQ_TRAIN_SIZE].copy()

t0 = time.time()
quantizer_pca = faiss.IndexFlatL2(DIM_PCA)
index_ivfpq_pca = faiss.IndexIVFPQ(quantizer_pca, DIM_PCA, IVFPQ_NLIST, IVFPQ_M_SUBQ, IVFPQ_NBITS)
index_ivfpq_pca.train(train_data_pca)
index_ivfpq_pca.add(base_pca)
build_time_ivfpq_pca = time.time() - t0

mem_index_ivfpq_pca = get_memory_mb() - mem_before

index_ivfpq_pca.nprobe = IVFPQ_NPROBE
latency_ivfpq_pca, qps_ivfpq_pca, I_ivfpq_pca = measure_search(index_ivfpq_pca, query_pca)
recall_ivfpq_pca = calculate_recall(I_ivfpq_pca, ground_truth)

print(f"  Build time : {build_time_ivfpq_pca:.2f}s")
print(f"  Mémoire    : {mem_index_ivfpq_pca:.1f} MB")
print(f"  Recall@10  : {recall_ivfpq_pca:.4f}")
print(f"  Latence    : {latency_ivfpq_pca:.4f} ms/query")
print(f"  QPS        : {qps_ivfpq_pca:.1f}")

del index_ivfpq_pca, quantizer_pca, train_data_pca
gc.collect()

print(f"\n📊 Delta (PCA vs original) :")
print(f"   Recall    : {recall_ivfpq_pca - recall_ivfpq:+.4f}  ({(recall_ivfpq_pca/recall_ivfpq - 1)*100:+.1f}%)")
print(f"   Latence   : {(latency_ivfpq_pca/latency_ivfpq - 1)*100:+.1f}%")
print(f"   Mémoire   : {(mem_index_ivfpq_pca/mem_index_ivfpq - 1)*100:+.1f}%")


## 7. Synthèse comparative

### 7.1 Tableau récapitulatif


In [ ]:
# ============================================
# TABLEAU COMPARATIF FINAL
# ============================================

results = pd.DataFrame([
    {'Méthode': 'HNSW',   'PCA': 'Non', 'Dim': DIM_ORIGINAL, 'Recall@10': recall_hnsw,
     'Latency (ms)': latency_hnsw, 'QPS': qps_hnsw,
     'Memory (MB)': mem_index_hnsw, 'Build (s)': build_time_hnsw},
    {'Méthode': 'HNSW',   'PCA': 'Oui', 'Dim': DIM_PCA, 'Recall@10': recall_hnsw_pca,
     'Latency (ms)': latency_hnsw_pca, 'QPS': qps_hnsw_pca,
     'Memory (MB)': mem_index_hnsw_pca, 'Build (s)': build_time_hnsw_pca},
    {'Méthode': 'LSH',    'PCA': 'Non', 'Dim': DIM_ORIGINAL, 'Recall@10': recall_lsh,
     'Latency (ms)': latency_lsh, 'QPS': qps_lsh,
     'Memory (MB)': mem_index_lsh, 'Build (s)': build_time_lsh},
    {'Méthode': 'LSH',    'PCA': 'Oui', 'Dim': DIM_PCA, 'Recall@10': recall_lsh_pca,
     'Latency (ms)': latency_lsh_pca, 'QPS': qps_lsh_pca,
     'Memory (MB)': mem_index_lsh_pca, 'Build (s)': build_time_lsh_pca},
    {'Méthode': 'IVF-PQ', 'PCA': 'Non', 'Dim': DIM_ORIGINAL, 'Recall@10': recall_ivfpq,
     'Latency (ms)': latency_ivfpq, 'QPS': qps_ivfpq,
     'Memory (MB)': mem_index_ivfpq, 'Build (s)': build_time_ivfpq},
    {'Méthode': 'IVF-PQ', 'PCA': 'Oui', 'Dim': DIM_PCA, 'Recall@10': recall_ivfpq_pca,
     'Latency (ms)': latency_ivfpq_pca, 'QPS': qps_ivfpq_pca,
     'Memory (MB)': mem_index_ivfpq_pca, 'Build (s)': build_time_ivfpq_pca},
])

results.to_csv(f'{RESULTS_PATH}pca_comparative_results.csv', index=False)

print("📊 Résultats complets :")
print(results.to_string(index=False))


In [ ]:
# Affichage stylé pour Jupyter
display(
    results.style
    .format({
        'Recall@10': '{:.4f}',
        'Latency (ms)': '{:.4f}',
        'QPS': '{:.1f}',
        'Memory (MB)': '{:.1f}',
        'Build (s)': '{:.2f}',
    })
    .background_gradient(subset=['Recall@10'], cmap='Greens')
    .background_gradient(subset=['Latency (ms)'], cmap='Reds_r')
    .background_gradient(subset=['Memory (MB)'], cmap='Blues_r')
)


### 7.2 Visualisations comparatives


In [ ]:
# ============================================
# GRAPHIQUE 1 : Recall avant/après PCA
# ============================================

methods = ['HNSW', 'LSH', 'IVF-PQ']
recall_no = [recall_hnsw, recall_lsh, recall_ivfpq]
recall_pca = [recall_hnsw_pca, recall_lsh_pca, recall_ivfpq_pca]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

x = np.arange(len(methods))
width = 0.35

# (a) Recall
axes[0].bar(x - width/2, recall_no, width, label='Sans PCA', color='#2E86AB')
axes[0].bar(x + width/2, recall_pca, width, label=f'PCA {DIM_PCA}D', color='#F18F01')
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, fontsize=12)
axes[0].set_ylabel('Recall@10', fontsize=12, fontweight='bold')
axes[0].set_title('Recall@10 : Avant / Après PCA', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)
for i, (n, p) in enumerate(zip(recall_no, recall_pca)):
    axes[0].text(i - width/2, n + 0.01, f'{n:.3f}', ha='center', fontsize=9)
    axes[0].text(i + width/2, p + 0.01, f'{p:.3f}', ha='center', fontsize=9)

# (b) Latence
lat_no = [latency_hnsw, latency_lsh, latency_ivfpq]
lat_pca = [latency_hnsw_pca, latency_lsh_pca, latency_ivfpq_pca]
axes[1].bar(x - width/2, lat_no, width, label='Sans PCA', color='#2E86AB')
axes[1].bar(x + width/2, lat_pca, width, label=f'PCA {DIM_PCA}D', color='#F18F01')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, fontsize=12)
axes[1].set_ylabel('Latence (ms/query)', fontsize=12, fontweight='bold')
axes[1].set_title('Latence : Avant / Après PCA', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

# (c) Mémoire
mem_no = [mem_index_hnsw, mem_index_lsh, mem_index_ivfpq]
mem_pca = [mem_index_hnsw_pca, mem_index_lsh_pca, mem_index_ivfpq_pca]
axes[2].bar(x - width/2, mem_no, width, label='Sans PCA', color='#2E86AB')
axes[2].bar(x + width/2, mem_pca, width, label=f'PCA {DIM_PCA}D', color='#F18F01')
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, fontsize=12)
axes[2].set_ylabel('Mémoire (MB)', fontsize=12, fontweight='bold')
axes[2].set_title('Empreinte mémoire : Avant / Après PCA', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}pca_impact_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================
# GRAPHIQUE 2 : Trade-off Recall vs Latence (scatter)
# ============================================

plt.figure(figsize=(11, 7))

colors = {'HNSW': '#2E86AB', 'LSH': '#A23B72', 'IVF-PQ': '#F18F01'}
markers_no = 'o'
markers_pca = 's'

for method, r_no, l_no, r_pca, l_pca in zip(
    methods,
    recall_no, lat_no,
    recall_pca, lat_pca
):
    # Sans PCA
    plt.scatter(l_no, r_no, s=250, marker=markers_no, color=colors[method],
                edgecolors='black', linewidths=1.5, label=f'{method} (128D)')
    # Avec PCA
    plt.scatter(l_pca, r_pca, s=250, marker=markers_pca, color=colors[method],
                edgecolors='black', linewidths=1.5, label=f'{method} (PCA 32D)')
    # Flèche
    plt.annotate('', xy=(l_pca, r_pca), xytext=(l_no, r_no),
                 arrowprops=dict(arrowstyle='->', color=colors[method], lw=1.5, alpha=0.5))

plt.xlabel('Latence (ms/query)', fontsize=13, fontweight='bold')
plt.ylabel('Recall@10', fontsize=13, fontweight='bold')
plt.title('Trade-off Recall vs Latence : impact de la PCA\n(◯ = sans PCA, □ = avec PCA 32D)',
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(loc='best', fontsize=10, ncol=2)
plt.xscale('log')

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}pca_tradeoff_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================
# GRAPHIQUE 3 : Synthèse des deltas (%)
# ============================================

deltas = pd.DataFrame({
    'Méthode': methods,
    'Δ Recall (%)': [(recall_hnsw_pca/recall_hnsw - 1)*100,
                     (recall_lsh_pca/recall_lsh - 1)*100 if recall_lsh > 0 else 0,
                     (recall_ivfpq_pca/recall_ivfpq - 1)*100],
    'Δ Latence (%)': [(latency_hnsw_pca/latency_hnsw - 1)*100,
                      (latency_lsh_pca/latency_lsh - 1)*100,
                      (latency_ivfpq_pca/latency_ivfpq - 1)*100],
    'Δ Mémoire (%)': [(mem_index_hnsw_pca/mem_index_hnsw - 1)*100 if mem_index_hnsw > 0 else 0,
                      (mem_index_lsh_pca/mem_index_lsh - 1)*100 if mem_index_lsh > 0 else 0,
                      (mem_index_ivfpq_pca/mem_index_ivfpq - 1)*100 if mem_index_ivfpq > 0 else 0],
})

fig, ax = plt.subplots(figsize=(12, 6))
deltas.set_index('Méthode').plot(kind='bar', ax=ax,
    color=['#27AE60', '#E74C3C', '#3498DB'], edgecolor='black', width=0.7)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_ylabel('Variation relative (%)', fontsize=12, fontweight='bold')
ax.set_title(f'Impact de la compression PCA ({DIM_ORIGINAL}D → {DIM_PCA}D) par méthode',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)

# Annoter les barres
for container in ax.containers:
    ax.bar_label(container, fmt='%+.1f%%', fontsize=9, padding=3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}pca_deltas_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Tableau des variations relatives :")
print(deltas.to_string(index=False))


## 8. Conclusions & Discussion

### 8.1 Résumé des observations

**Coût offline de la PCA** : la PCA n'est entraînée qu'une fois, son coût est amorti sur tous les usages ultérieurs.

**Observations attendues (à confirmer empiriquement) :**

1. **HNSW** : léger recul du recall (HNSW gérait déjà bien la haute dimension), mais gain en mémoire et latence.

2. **LSH** : **gros gain de recall** 📈. LSH souffrait particulièrement du "curse of dimensionality" — la PCA le rend nettement plus compétitif. C'est le résultat le plus spectaculaire du benchmark.

3. **IVF-PQ** : effet modéré sur le recall (IVF-PQ a déjà sa propre compression via la quantification). La PCA reste bénéfique en mémoire et build time.

### 8.2 Recommandations d'usage

| Contexte | Choix recommandé |
|---|---|
| **Recall prioritaire, RAM OK** | HNSW (sans PCA) |
| **RAM très limitée** | LSH + PCA ou IVF-PQ + PCA |
| **Très grande échelle (>10M vecteurs)** | IVF-PQ (avec PCA optionnelle) |
| **Dimension très élevée (>500)** | PCA quasi-obligatoire pour LSH |

### 8.3 Conclusion générale

La compression PCA est un **complément orthogonal** aux méthodes ANN : elle peut s'appliquer avant n'importe quel index et apporte systématiquement des gains en mémoire et en vitesse. Son impact sur la précision dépend fortement de la méthode sous-jacente :
- **Bénéfique pour LSH** (qui souffre de la haute dimension)
- **Neutre à légèrement négatif pour HNSW** (déjà robuste)
- **Neutre pour IVF-PQ** (redondance partielle des compressions)

**Le meilleur pipeline dépend du triangle des contraintes** : Recall / Latence / Mémoire.


In [ ]:
# ============================================
# SAUVEGARDE FINALE DES RÉSULTATS
# ============================================

print("="*60)
print("FICHIERS GÉNÉRÉS")
print("="*60)

generated = [
    f'{RESULTS_PATH}pca_comparative_results.csv',
    f'{RESULTS_PATH}pca_variance_explained.png',
    f'{RESULTS_PATH}pca_impact_all_methods.png',
    f'{RESULTS_PATH}pca_tradeoff_scatter.png',
    f'{RESULTS_PATH}pca_deltas_summary.png',
]
for g in generated:
    if os.path.exists(g):
        size_kb = os.path.getsize(g) / 1024
        print(f"  ✓ {g}  ({size_kb:.1f} KB)")

print("\n✅ Benchmark comparatif terminé !")
